# Struct2Seq-GCN: Inference Demonstration
Here we demonstrate the complete predictive pipeline for the **Heterogeneous Graph Architecture**. We load the trained weights and execute `scripts/inference.py` to predict an amino acid sequence from a raw Protein-Ligand PDB file.

In [8]:
import sys
import os
import subprocess
import shutil

# Check if we are running in Google Colab (remote Linux VM)
if 'google.colab' in sys.modules:
    print("Detected Google Colab environment. Setting up via GitHub...")
    
    # 1. Install required packages using PyG pre-compiled wheels to avoid long build times
    import torch
    torch_version = torch.__version__.split('+')[0]
    # Format cuda version, e.g. '12.1' -> 'cu121'
    if torch.version.cuda:
        cuda_version = f"cu{torch.version.cuda.replace('.', '')}"
    else:
        cuda_version = "cpu"
        
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html"
    
    print(f"Installing PyG dependencies from {wheel_url}...")
    subprocess.run(["pip", "install", "-q", "torch_geometric", "prody"], check=True)
    subprocess.run(["pip", "install", "-q", "torch-cluster", "-f", wheel_url], check=True)

    # 2. Clone/update main repository
    repo_url = "https://github.com/WSobo/Struct2Seq-GCN.git"
    target_dir = "/content/Struct2Seq-GCN"

    if not os.path.exists(target_dir):
        os.chdir('/content')
        print("Cloning repository...")
        subprocess.run(["git", "clone", repo_url], check=True)
    else:
        os.chdir(target_dir)
        print("Pulling latest changes...")
        subprocess.run(["git", "fetch", "--all"], check=True)
        subprocess.run(["git", "pull"], check=True)

    # 3. Ensure LigandMPNN dependency exists (fallback for gitlink/submodule issues)
    ligand_dir = os.path.join(target_dir, "LigandMPNN")
    ligand_data_utils = os.path.join(ligand_dir, "data_utils.py")
    if not os.path.exists(ligand_data_utils):
        print("Fetching LigandMPNN dependency...")
        if os.path.exists(ligand_dir):
            shutil.rmtree(ligand_dir)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/dauparas/LigandMPNN.git", "LigandMPNN"],
            cwd=target_dir,
            check=True,
        )

    # 4. Enter the project directory
    os.chdir(target_dir)
    print("Colab setup complete. Current Dir:", os.getcwd())

Detected Google Colab environment. Setting up via GitHub...
Installing PyG dependencies from https://data.pyg.org/whl/torch-2.10.0+cu128.html...
Pulling latest changes...
Colab setup complete. Current Dir: /content/Struct2Seq-GCN


In [25]:
#@title Which Branch?

!git fetch --all
!git checkout feat/advanced-gnn-features
!git pull origin feat/advanced-gnn-features

Fetching origin
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 13 (delta 6), reused 12 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 97.07 KiB | 795.00 KiB/s, done.
From https://github.com/WSobo/Struct2Seq-GCN
   1d26dbb..a0c8f1b  main       -> origin/main
 * [new branch]      feat/advanced-gnn-features -> origin/feat/advanced-gnn-features
M	LigandMPNN
Branch 'feat/advanced-gnn-features' set up to track remote branch 'feat/advanced-gnn-features' from 'origin'.
Switched to a new branch 'feat/advanced-gnn-features'
From https://github.com/WSobo/Struct2Seq-GCN
 * branch            feat/advanced-gnn-features -> FETCH_HEAD
Already up to date.


In [22]:
import os

# Ensure we operate from the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/Struct2Seq-GCN


## 1. Quick Syntax and Inference Logic Check
We can utilize our newly saved script `inference.py` to target an existing sample structure that contains a ligand. We'll output the generated sequence to a FASTA file.

In [27]:
# Set paths for a local demonstration
import subprocess

pdb_path = "LigandMPNN/inputs/1BC8.pdb" 
weights_path = "outputs/best_model.pt"
out_fasta = "outputs/1BC8_predicted.fasta"

# Optionally set PYTHONPATH if you're running in Colab to ensure module visibility
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:{os.getcwd()}"

cmd = [
    "python", "scripts/inference.py",
    "--pdb", pdb_path,
    "--weights", weights_path,
    "--out_fasta", out_fasta
]

print(f"Executing: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

print("--- STDOUT ---")
print(result.stdout)

if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)

Executing: python scripts/inference.py --pdb LigandMPNN/inputs/1BC8.pdb --weights outputs/best_model.pt --out_fasta outputs/1BC8_predicted.fasta
--- STDOUT ---
Parsing coordinates from LigandMPNN/inputs/1BC8.pdb...
Graph Built: 93 Protein Nodes, 406 Ligand Nodes.
Loading trained weights from outputs/best_model.pt...
Running inference...

INFERENCE RESULTS
Native Sequence    : MDSAITLWQFLLQLLQKPQNKHMICWTSNDGQFKLLQAEEVARLWGIRKNKPNMNYDKLSRALRYYYVKNIIKKVNGQKFVYKFVSYPEILNM
Predicted Sequence : EGDDLSLLELLKLLLKDPELLELAIETSDDGVFTLPDPKELAKELAKKLNIPNADKKLRRRILRLLKKKGLVYRRPNRAGKYTLPLYPALLHP
--------------------------------------------------
Native Sequence Recovery (NSR): 25.81% (24/93 residues)

Saved FASTA to outputs/1BC8_predicted.fasta



## 2. Inspect Generated Outputs
Read the generated FASTA output file to visually inspect the final string output emitted by the pipeline.

In [28]:
if os.path.exists(out_fasta):
    with open(out_fasta, 'r') as f:
        print(f.read())
else:
    print(f"Error: {out_fasta} not found.")

>Struct2SeqGCN_Predicted_1BC8
EGDDLSLLELLKLLLKDPELLELAIETSDDGVFTLPDPKELAKELAKKLNIPNADKKLRRRILRLLKKKGLVYRRPNRAGKYTLPLYPALLHP

